# Week 25, Day 3: How many dimensions? Reconstruction and cross-validation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w025/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
data = np.load(dataset("churchland2012_reach.npz"))
rates, times = data["rates"].astype(float), data["times"]
C, T, N = rates.shape
Xn = rates / (rates.max(axis=(0, 1)) - rates.min(axis=(0, 1)) + 5)
X = Xn - Xn.mean(axis=0, keepdims=True)
win = (times >= -50) & (times <= 150)
Xw = X[:, win, :]
D = Xw.reshape(-1, N); mu = D.mean(axis=0)
U, S, Vt = np.linalg.svd(D - mu, full_matrices=False)
print("ready:", D.shape)

### Exercise: Rebuilding each neuron from six components

Write `reconstruct(k)`, which returns the rank-k reconstruction of D. Compute each neuron's R² for k = 6 in `r2_neuron` (one minus its residual sum of squares over its total sum of squares about its mean) and the median `median_r2`.

In [ ]:
def reconstruct(k):
    Vk = Vt[:k].T
    return ...

Dh = reconstruct(6)
r2_neuron = ...
median_r2 = np.median(r2_neuron)
print(f"rank 6: median neuron R2 {median_r2:.2f};  a quarter of neurons below {np.percentile(r2_neuron, 25):.2f}")
best = np.argmax(np.ptp(D, axis=0))
plt.plot(D[:21, best], label="data, condition 1"); plt.plot(Dh[:21, best], "--", label="6 components")
plt.legend(); plt.xlabel("time bin"); plt.show()

In [ ]:
import numpy as np
def test_r2():
    "r2_neuron holds each neuron's R squared for the rank-6 reconstruction"
    V6 = Vt[:6].T; Dh = mu + (D - mu) @ V6 @ V6.T
    ref = 1 - ((D - Dh) ** 2).sum(0) / ((D - mu) ** 2).sum(0)
    assert r2_neuron.shape == (N,) and np.allclose(r2_neuron, ref)

def test_median():
    "median_r2 is the median over neurons"
    assert abs(median_r2 - np.median(r2_neuron)) < 1e-12

test_r2()
test_median()
print("All checks passed")

### Exercise: Choosing k on held-out neurons and conditions

Complete `bicv(k)`: for each of 10 random splits (local rng seed 7) into halves of conditions (A, B) and neurons (n1, n2), fit PCA on the training block (A, n1), get latents for A and B by projecting their n1 activity, fit a least-squares regression (with an intercept) from latents to n2 on A, and score the prediction of n2 on B by R². Return the mean. Evaluate it for every k in `ks`.

In [ ]:
def bicv(k, reps=10):
    r = np.random.default_rng(7); out = []
    for _ in range(reps):
        cp, npm = r.permutation(C), r.permutation(N)
        A, B, n1, n2 = cp[:C // 2], cp[C // 2:], npm[:N // 2], npm[N // 2:]
        DA1 = Xw[A][:, :, n1].reshape(-1, len(n1)); m1 = DA1.mean(0)
        Vk = np.linalg.svd(DA1 - m1, full_matrices=False)[2][:k].T
        LA = (DA1 - m1) @ Vk
        LB = (Xw[B][:, :, n1].reshape(-1, len(n1)) - m1) @ Vk
        YA = Xw[A][:, :, n2].reshape(-1, len(n2)); YB = Xw[B][:, :, n2].reshape(-1, len(n2))
        W = ...                                   # regression from [LA, 1] to YA
        pred = np.c_[LB, np.ones(len(LB))] @ W
        out.append(...)                           # R squared of pred on YB
    return np.mean(out)

ks = [1, 2, 4, 6, 8, 10, 15, 20, 30, 40, 60, 80]
scores = [bicv(k) for k in ks]
print({k: round(float(s), 3) for k, s in zip(ks, scores)})
plt.plot(ks, scores, "o-"); plt.xlabel("k"); plt.ylabel("held-out R2"); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "scores holds a mean held-out R squared for every k in ks"
    assert len(scores) == len(ks) and all(-1 < s < 1 for s in scores)

def test_peak():
    "the score rises from k = 1, peaks in the middle of the range and falls again by k = 80"
    i = int(np.argmax(scores))
    assert scores[0] < scores[i] and 10 <= ks[i] <= 50 and scores[-1] < scores[i]

test_shape()
test_peak()
print("All checks passed")

### Exercise: Early and late subspaces

Write `top_dirs(mask, k=4)`, the top k principal directions (as columns) of X restricted to the times in mask. Compute `align`: the late window's variance captured by the early directions divided by the variance captured by its own top four. Compute `align_random` the same way with four random orthonormal directions (local rng seed 0, QR of a normal matrix).

In [ ]:
def top_dirs(mask, k=4):
    Dm = X[:, mask].reshape(-1, N); Dm = Dm - Dm.mean(0)
    return np.linalg.svd(Dm, full_matrices=False)[2][:k].T

early, late = (times >= -50) & (times <= 40), (times >= 100) & (times <= 190)
DL = X[:, late].reshape(-1, N); DL = DL - DL.mean(0)
own = np.sum(np.linalg.svd(DL, compute_uv=False)[:4] ** 2)
align = ...
Q = np.linalg.qr(np.random.default_rng(0).normal(size=(N, 4)))[0]
align_random = ...
print(f"alignment index early to late: {align:.2f};  random directions: {align_random:.2f}")

In [ ]:
import numpy as np
def test_range():
    "the alignment index lies between 0 and 1"
    assert 0 < align < 1 and 0 < align_random < 1

def test_compare():
    "the early subspace captures much less of the late activity than the late subspace does (index below 0.5), but far more than random directions"
    assert align < 0.5 and align > 2 * align_random

test_range()
test_compare()
print("All checks passed")